# Patrón Estructural: Adapter

## Dominio del ejemplo: sistema bancario

## Introducción
El patrón **Adapter** permite que dos interfaces **incompatibles** trabajen juntas.
Envuelve un objeto existente con una **nueva interfaz** que el cliente sí entiende.

### ¿Qué problema resuelve en la banca?
Nuestro banco consulta el riesgo de un cliente mediante una interfaz propia
`ConsultaRiesgo.evaluar(documento) -> dict`. Contratamos un **buró de crédito externo**
(DataCredito) cuya librería expone otro método y otro formato:
`DataCreditoAPI.obtener_score(cedula) -> str` (devuelve un texto). No podemos (ni
debemos) modificar la librería del proveedor. El Adapter **traduce** entre ambas.

## Código *sin patrón* (el problema es evidente)
El sistema del banco espera un objeto con `evaluar(...)`, pero el proveedor externo tiene
otra firma y otro formato de salida. Llamarlo directamente **rompe**.

In [1]:
# Interfaz que el resto del banco YA usa en todas partes
class MotorRiesgoBanco:
    def evaluar(self, documento: str) -> dict:
        # Devuelve un dict estandar del banco
        return {"documento": documento, "score": 750, "aprobado": True}


# Libreria del proveedor externo: OTRA interfaz, NO la podemos cambiar
class DataCreditoAPI:
    def obtener_score(self, cedula: str) -> str:
        # Devuelve un texto plano, no un dict
        return f"cedula={cedula};puntaje=620;estado=REVISION"


def analizar_cliente(motor, documento):
    # El cliente asume la interfaz del banco: motor.evaluar(...)
    return motor.evaluar(documento)


print(analizar_cliente(MotorRiesgoBanco(), "1020304050"))

# Si intentamos usar el proveedor externo con el mismo codigo, falla:
try:
    print(analizar_cliente(DataCreditoAPI(), "1020304050"))
except AttributeError as e:
    print(">> Problema: AttributeError ->", e)
    print(">> El proveedor no tiene 'evaluar' y su salida es texto, no dict.")

{'documento': '1020304050', 'score': 750, 'aprobado': True}
>> Problema: AttributeError -> 'DataCreditoAPI' object has no attribute 'evaluar'
>> El proveedor no tiene 'evaluar' y su salida es texto, no dict.


### Análisis del problema
- `DataCreditoAPI` **no** tiene el método `evaluar` que el banco usa en todos lados.
- Aunque lo tuviera, devuelve **texto** y el banco espera un **dict** estandarizado.
- Modificar cada punto del banco para tratar al proveedor "distinto" ensuciaría todo el
  código y lo ataría a un proveedor concreto.

## Código *con patrón* (problema resuelto)
Creamos `AdaptadorDataCredito` que **implementa la interfaz del banco** (`evaluar`) y por
dentro llama al proveedor externo y **traduce** su salida al formato estándar.

In [2]:
from abc import ABC, abstractmethod


# Interfaz objetivo (la que el banco entiende)
class ConsultaRiesgo(ABC):
    @abstractmethod
    def evaluar(self, documento: str) -> dict: ...


# Implementacion propia del banco
class MotorRiesgoBanco(ConsultaRiesgo):
    def evaluar(self, documento: str) -> dict:
        return {"documento": documento, "score": 750, "aprobado": True}


# Servicio externo incompatible (Adaptee) - NO se modifica
class DataCreditoAPI:
    def obtener_score(self, cedula: str) -> str:
        return f"cedula={cedula};puntaje=620;estado=REVISION"


# ADAPTER: implementa la interfaz del banco y traduce al proveedor externo
class AdaptadorDataCredito(ConsultaRiesgo):
    def __init__(self, api: DataCreditoAPI):
        self._api = api

    def evaluar(self, documento: str) -> dict:
        crudo = self._api.obtener_score(documento)          # "cedula=...;puntaje=620;estado=REVISION"
        partes = dict(p.split("=") for p in crudo.split(";"))
        score = int(partes["puntaje"])
        return {
            "documento": partes["cedula"],
            "score": score,
            "aprobado": partes["estado"] == "APROBADO" or score >= 700,
        }


def analizar_cliente(motor: ConsultaRiesgo, documento: str) -> dict:
    return motor.evaluar(documento)  # el cliente NO cambia


# Ahora ambos se usan EXACTAMENTE igual
print("Motor propio :", analizar_cliente(MotorRiesgoBanco(), "1020304050"))
print("Buro externo :", analizar_cliente(AdaptadorDataCredito(DataCreditoAPI()), "1020304050"))
print(">> Solucion: el proveedor externo se usa con la MISMA interfaz del banco.")

Motor propio : {'documento': '1020304050', 'score': 750, 'aprobado': True}
Buro externo : {'documento': '1020304050', 'score': 620, 'aprobado': False}
>> Solucion: el proveedor externo se usa con la MISMA interfaz del banco.


### Verificación
- `AdaptadorDataCredito` expone `evaluar(...)` igual que el motor propio.
- El cliente `analizar_cliente` funciona **sin cambios** con ambos.
- La salida del proveedor (texto) se **tradujo** al dict estándar del banco.

## UML del patrón Adapter
```plantuml
@startuml
interface ConsultaRiesgo {
    + evaluar(documento) : dict
}
class MotorRiesgoBanco {
    + evaluar(documento) : dict
}
class AdaptadorDataCredito {
    - _api : DataCreditoAPI
    + evaluar(documento) : dict
}
class DataCreditoAPI {
    + obtener_score(cedula) : str
}
ConsultaRiesgo <|.. MotorRiesgoBanco
ConsultaRiesgo <|.. AdaptadorDataCredito
AdaptadorDataCredito --> DataCreditoAPI : adapta
@enduml
```

## ¿Por qué Adapter y no otro patrón?
- El problema es de **incompatibilidad de interfaces** entre un servicio existente que no
  podemos modificar (el buró externo) y el contrato que usa el banco. Ese es el caso de
  uso textual de Adapter.
- No es Facade (que **simplifica** un subsistema complejo, no traduce una interfaz), ni
  Decorator (que **añade responsabilidades** manteniendo la misma interfaz). Aquí no
  añadimos funciones ni simplificamos: **convertimos** una interfaz en otra.
- Adapter nos deja cambiar de proveedor mañana escribiendo otro adaptador, sin tocar el
  resto del banco.